In [ ]:

import tensorflow as tf
from tensorflow import keras
import numpy as np
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset, random_split, Subset

import kagglehub
import matplotlib.pyplot as plt


## Seed = 42

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


# Exercise 1
## Training Deep Neural Networkon MNIST

### Data preprocessing

In [ ]:
def create_dataset(kagglehub_path, normalizer, train_size, test_size, batch_size, dataset=datasets.MNIST):
    path = kagglehub.dataset_download(kagglehub_path)

    transform = transforms.Compose([
        transforms.ToTensor(),
        normalizer
    ])

    train_data_raw = dataset(root=path, train=True, download=True, transform=transform)
    test_data_raw = dataset(root=path, train=False, download=True, transform=transform)

    train_subset = Subset(train_data_raw, range(train_size))
    test_subset = Subset(test_data_raw, range(test_size))

    ts = int(train_size*0.8)
    vs = train_size - ts
    train_data, val_data = random_split(train_subset, (ts, vs), generator=torch.Generator())

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader


In [ ]:
train_loader, val_loader, test_laoder = create_dataset(
    kagglehub_path="coderanand/mnist-dataset",
    normalizer=transforms.Normalize((0.1307,), (0.3081,)),
    train_size=1000,
    test_size=200,
    batch_size=32
)

### Neural network

In [ ]:
class MNISTModel(nn.Module):
    def __init__(self):
        super(MNISTModel, self).__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 64),
            nn.ELU(),
            nn.Linear(64, 64), 
            nn.ELU(),
            nn.Linear(64, 64),
            nn.ELU(),
            nn.Linear(64, 10),
            nn.LogSoftmax(dim=1)
        )
        self.init_weights()
    
    def init_weights(self):
        for layer in self.layers:
            if isinstance(layer, nn.Linear): # Skip ELU
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu") # He init
                # https://www.geeksforgeeks.org/deep-learning/kaiming-initialization-in-deep-learning/
                nn.init.zeros_(layer.bias)
        # Initialize output layer with Xavier
        #nn.init.xavier_uniform_(self.layers[-2].weight)
    
    def forward(self, x):
        return self.layers(x)
    

model = MNISTModel()

### Optimizer and criterion

In [ ]:
optimizer = torch.optim.NAdam(model.parameters(), lr=0.001) # Optimizer and lr
criterion = nn.CrossEntropyLoss()

### Training with early stopping

In [ ]:
def fit_model(model, epochs, patience, train_losses, val_losses, train_acc, val_acc):
    global train_loader
    global val_loader
    global optimizer
    global criterion

    best_val_loss = np.inf
    best_model_state: dict = None
    patience_counter = 0

    for epoch in range(epochs):
        # Training
        model.train()
        correct_train = 0
        total_train = 0
        train_loss = 0

        for X, y in train_loader:
            optimizer.zero_grad() # Reset gradients
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

            # Accuracy
            _, preds = torch.max(outputs, 1)
            correct_train += (preds == y).sum().item()
            total_train += y.size(0)

        # Validation
        model.eval()
        val_loss = 0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for X, y in val_loader:
                outputs = model(X)
                loss = criterion(outputs, y)
                val_loss += loss.item()

                # Accuracy
                _, preds = torch.max(outputs, 1)
                correct_val += (preds == y).sum().item()
                total_val += y.size(0)
        
        

        # Averages
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        train_accuracy = correct_train / total_train
        val_accuracy = correct_val / total_val

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_acc.append(train_accuracy)
        val_acc.append(val_accuracy)

        # Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch} epochs.")
                break
        
    return best_model_state

epochs = 50
patience = 5
train_losses, val_losses = [], []
train_acc, val_acc = [], []

best_model_state = fit_model(model, epochs, patience, train_losses, val_losses, train_acc, val_acc)

model.load_state_dict(best_model_state)

### Test accuracy

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss', color='salmon', marker="o")
plt.plot(val_losses, label='Validation Loss', color='skyblue', marker="o")
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(train_acc, label='Training Accuracy', color='salmon', marker="o")
plt.plot(val_acc, label='Validation Accuracy', color='skyblue', marker="o")
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Exercise 2
## Training Deep Neural Network on CIFAR-10

### Data preprocessing

In [ ]:
cifar_path = kagglehub.dataset_download("hamadmo/cifar10")

train_loader, val_loader, test_loader = create_dataset(
    kagglehub_path="hamadmo/cifar10",
    normalizer=transforms.Normalize((0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)),
    train_size=4000,
    test_size=1000,
    batch_size=128,
    dataset=datasets.CIFAR10
)

### Model

In [ ]:
class CIFARModel(nn.Module
    def __init__(self):
        super(CIFARModel, self).__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 256),
            nn.ELU(),
            nn.Linear(256, 256), 
            nn.ELU(),
            nn.Linear(256, 256),
            nn.ELU(),
            nn.Linear(256, 256),
            nn.ELU(),
            nn.Linear(256, 10),
            nn.LogSoftmax(dim=1)
        )
        self.init_weights()
    
    def init_weights(self):
        for layer in self.layers:
            if isinstance(layer, nn.Linear): # Skip ELU
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu") # He init
                nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        return self.layers(x)
    

CIFARModel = CIFARModel()

### Optimizer and criterion

In [ ]:
optimizer = torch.optim.NAdam(CIFARModel.parameters(), lr=0.001) # Optimizer and lr
criterion = nn.CrossEntropyLoss()

In [ ]:
epochs = 50
patience = 5
train_losses, val_losses = [], []
train_acc, val_acc = [], []

best_model_state = fit_model(
    CIFARModel,
    epochs=50,
    patience=5,
    train_losses=train_losses,
    val_losses=val_losses,
    train_acc=train_acc,
    val_acc=val_acc
)

CIFARModel.load_state_dict(best_model_state)

In [ ]:
def test_accuracy(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in test_loader:
            preds = model(X).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += len(y)
    return correct / total

alpha_dropout_acc = test_accuracy(CIFARModel)
print("Q2.1 Test accuracy", alpha_dropout_acc)

### Plotting train and validation loss

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss', color='salmon', marker="o")
plt.plot(val_losses, label='Validation Loss', color='skyblue', marker="o")
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(train_acc, label='Training Accuracy', color='salmon', marker="o")
plt.plot(val_acc, label='Validation Accuracy', color='skyblue', marker="o")
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Exercise 3
## Regularization with Alpha Dropout and MC Dropout

### Data preprocessing

In [ ]:
train_loader, val_loader, test_loader = create_dataset(
    kagglehub_path="coderanand/mnist-dataset",
    normalizer=transforms.Normalize((0.1307,), (0.3081,)),
    train_size=1000,
    test_size=200,
    batch_size=32,
    dataset=datasets.MNIST
)


In [ ]:
class MNIST_SELU_AlphaDropout(nn.Module):
    def __init__(self):
        super(MNIST_SELU_AlphaDropout, self).__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(784, 64)
        self.drop1 = nn.AlphaDropout(0.1)
        self.hidden2 = nn.Linear(64, 64)
        self.drop2 = nn.AlphaDropout(0.1)
        self.hidden3 = nn.Linear(64, 64)
        self.drop3 = nn.AlphaDropout(0.1)
        self.output = nn.Linear(64, 10)

        self.init_weights()

    def init_weights(self):
        for layer in [self.hidden1, self.hidden2, self.hidden3, self.output]:
            nn.init.uniform_(layer.weight, 
                             -np.sqrt(1/ layer.in_features), 
                              np.sqrt(1/ layer.in_features))  # LeCun normal approx
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        x = self.flatten(x)
        x = F.selu(self.hidden1(x))
        x = self.drop1(x)
        x = F.selu(self.hidden2(x))
        x = self.drop2(x)
        x = F.selu(self.hidden3(x))
        x = self.drop3(x)
        return F.log_softmax(self.output(x), dim=1)
    

In [ ]:
selu_model = MNIST_SELU_AlphaDropout()

optimizer = torch.optim.NAdam(selu_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

train_losses, val_losses, train_acc, val_acc = [], [], [], []
best_model_state = fit_model(selu_model, epochs, patience, train_losses, val_losses, train_acc, val_acc)

selu_model.load_state_dict(best_model_state)

In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(train_losses, label="Train Loss"); plt.plot(val_losses,label="Val Loss")
plt.title("Loss"); plt.legend(); plt.grid()

plt.subplot(1,2,2)
plt.plot(train_acc, label="Train Acc"); plt.plot(val_acc,label="Val Acc")
plt.title("Accuracy"); plt.legend(); plt.grid()
plt.show()


In [ ]:
def test_accuracy(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in test_loader:
            preds = model(X).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += len(y)
    return correct / total

alpha_dropout_acc = test_accuracy(selu_model)
print("Q3.1 Test accuracy with Alpha Dropout:", alpha_dropout_acc)


In [ ]:
def mc_dropout_predict(model, X, passes=20):
    model.train()  # enable dropout
    preds = []
    with torch.no_grad():
        for _ in range(passes):
            preds.append(model(X))
    return torch.stack(preds).mean(dim=0)

def mc_dropout_test(model):
    correct = 0
    total = 0
    for X, y in test_loader:
        outputs = mc_dropout_predict(model, X, passes=20)
        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += len(y)
    return correct / total

mc_acc = mc_dropout_test(selu_model)
print("Q3.2 MC Dropout enhanced accuracy:", mc_acc)


In [ ]:
plt.figure(figsize=(5,4))
plt.bar(["Alpha Dropout", "MC Dropout"], [alpha_dropout_acc, mc_acc])
plt.ylabel("Accuracy")
plt.title("Test Accuracy Comparison")
plt.ylim(0,1)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt
import torch

# Ensure model is in evaluation mode first
selu_model.eval()

# Get a random batch from test_loader
batch = next(iter(test_loader))
X_batch_full, y_batch_full = batch

# Pick a random sample from the batch
idx = random.randint(0, X_batch_full.size(0) - 1)
X = X_batch_full[idx].unsqueeze(0)  # add batch dimension
y_true = y_batch_full[idx].item()

# MC Dropout predictions (20 stochastic forward passes)
preds = []
with torch.no_grad():
    for _ in range(20):
        selu_model.train()  # keep AlphaDropout active during inference
        preds.append(selu_model(X))
preds = torch.stack(preds).squeeze().exp()  # log-softmax -> probabilities

# Compute mean and std
mean_probs = preds.mean(dim=0)
std_probs = preds.std(dim=0)
y_pred = mean_probs.argmax().item()

# Plot image and MC Dropout probabilities
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(X.squeeze(), cmap='gray')
plt.title(f"True: {y_true}, Pred: {y_pred}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.bar(range(10), mean_probs, yerr=std_probs, capsize=5)
plt.title("MC-Dropout Class Probabilities\n(Mean ± Std)")
plt.xlabel("Digit class")
plt.ylabel("Probability")
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


# Exercise 4
## Transfer Learning with Pre-trained CNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, random_split
import numpy as np
import random

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Scaled to match input shape of MobileNetV2
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

full_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
full_test = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# Use only first 2000 train and 500 test samples
train_subset = Subset(full_train, range(2000))
test_subset = Subset(full_test, range(500))

# Split train_subset into training and validation (80-20)
train_size = int(0.8*len(train_subset))
val_size = len(train_subset) - train_size
train_data, val_data = random_split(train_subset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))

BATCH_SIZE = 32
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
for param in mobilenet.parameters():
    param.requires_grad = False

# Replace classifier
num_features = mobilenet.last_channel  # 1280
mobilenet.classifier = nn.Sequential(
    nn.Linear(num_features, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 10)
)

# Move to device
mobilenet = mobilenet.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mobilenet.parameters(), lr=0.001)

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):
    mobilenet.train()
    train_loss = 0
    correct = 0
    total = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = mobilenet(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    train_acc = correct/total

    # Validation
    mobilenet.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            outputs = mobilenet(X)
            loss = criterion(outputs, y)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct_val += (preds == y).sum().item()
            total_val += y.size(0)
    val_acc = correct_val / total_val
    print(f"Epoch {epoch+1}/{EPOCHS}: Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")


In [ ]:
mobilenet.eval()
correct_test = 0
total_test = 0
with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        outputs = mobilenet(X)
        _, preds = torch.max(outputs, 1)
        correct_test += (preds == y).sum().item()
        total_test += y.size(0)

test_acc = correct_test / total_test
print("Q4.1 Test accuracy:", test_acc)

# Exercise 5

In [ ]:
from torchvision.datasets import SVHN

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),  # convert to [0,1]
])

# SVHN dataset
train_full = SVHN(root="./data", split='train', download=True, transform=transform)
test_full = SVHN(root="./data", split='test', download=True, transform=transform)

# Use only first 2000 train samples and first 500 test samples
train_subset = Subset(train_full, range(2000))
test_subset = Subset(test_full, range(500))

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

In [ ]:
class DeepCNN(nn.Module):
    def __init__(self):
        super(DeepCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # Conv2D 32
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), # Conv2D 32
            nn.ReLU(),
            nn.MaxPool2d(2,2),                           # MaxPooling 2x2

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # Conv2D 64
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), # Conv2D 64
            nn.ReLU(),
            nn.MaxPool2d(2,2)                            # MaxPooling 2x2
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 256),                      # Flatten -> Dense 256
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)                           # Output layer
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
model = DeepCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    train_acc = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f}")


In [ ]:
model.eval()
correct_test = 0
total_test = 0
with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        _, preds = torch.max(outputs, 1)
        correct_test += (preds == y).sum().item()
        total_test += y.size(0)

test_acc = correct_test / total_test
print("Q5.1 Test accuracy:", test_acc)

# Exercise 6

In [ ]:
transform = transforms.ToTensor()

train_full = SVHN(root="./data", split='train', download=True, transform=transform)
test_full = SVHN(root="./data", split='test', download=True, transform=transform)

train_subset = Subset(train_full, range(2000))
test_subset = Subset(test_full, range(500))

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

In [ ]:
class MC_Dropout_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), nn.ReLU(),
            nn.Dropout(0.25),  # keep during test for MC Dropout
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
model = MC_Dropout_CNN().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()


In [ ]:
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    total, correct = 0, 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        _, pred = out.max(1)
        correct += pred.eq(y).sum().item()
        total += y.size(0)

    print(f"Epoch {epoch+1}: Train Acc = {correct/total:.4f}")

In [ ]:
def evaluate_plain():
    model.eval()  # dropout disabled
    total, correct = 0, 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            _, pred = out.max(1)
            correct += pred.eq(y).sum().item()
            total += y.size(0)
    return correct/total

plain_acc = evaluate_plain()

In [ ]:
def mc_dropout_predict(x, T=20):
    model.train()  # IMPORTANT: enable dropout in inference
    preds = []
    with torch.no_grad():
        for _ in range(T):
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            preds.append(probs.unsqueeze(0))
    preds = torch.cat(preds, dim=0)
    mean_pred = preds.mean(dim=0)
    var_pred = preds.var(dim=0).mean(dim=1)  # predictive variance per sample
    return mean_pred, var_pred

mc_correct = 0
uncertainties = []

for X, y in test_loader:
    X, y = X.to(device), y.to(device)
    mean_pred, var_pred = mc_dropout_predict(X, T=20)
    _, pred = mean_pred.max(1)
    mc_correct += pred.eq(y).sum().item()
    uncertainties.append(var_pred.cpu())

mc_acc = mc_correct / len(test_subset)
avg_uncertainty = torch.cat(uncertainties).mean().item()

print("\nQ6.1 Plain test accuracy:", round(plain_acc, 4))
print("Q6.2 MC Dropout test accuracy:", round(mc_acc, 4))
print("Q6.3 Mean epistemic uncertainty:", round(avg_uncertainty, 3))
